What is Structured Output in LangChain?


Structured output in LangChain means instructing an LLM to return its response in a predefined format instead of returning ordinary text.

For example, instead of asking an LLM about a person and receiving a paragraph, you can make it return specific fields such as name, age, and city in a structured format.

Normal output:

Input:
Tell me about Rahul, who is 25 years old
and lives in Hyderabad

OutPut Normal :

Rahul is a 25-year-old person who lives
in Hyderabad.


Structured json output:

{
    "name": "Rahul",
    "age": 25,
    "city": "Hyderabad"
}

What is Pydantic in Python?

Pydantic is a Python library used to validate data and ensure that it follows a predefined structure and data types.

In LangChain, we commonly use Pydantic to tell an LLM what fields it should return and what type of data each field should contain.

For example, if you expect a person's name to be a string and age to be an integer, Pydantic helps validate those requirements.

In [2]:

import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model("groq:openai/gpt-oss-120b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.11', 'langchain': '1.4.6'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002BBF21FA060>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002BBF0D11FA0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was realsed")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie")


In [6]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.11', 'langchain': '1.4.6'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002BBF21FA060>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002BBF0D11FA0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The t

In [8]:
model.invoke("Provide details about the movie RRR")

AIMessage(content='**RRR (Rise Roar Revolt)**  \n\n| Item | Details |\n|------|---------|\n| **Full Title** | *RRR* (stylized in all‑caps) |\n| **Release Date** | 13 March\u202f2022 (India) – worldwide theatrical releases continued through 2022‑2023 |\n| **Country** | India |\n| **Language** | Primarily Telugu (original), dubbed in Hindi, Tamil, Malayalam, Kannada, and later released in several international languages (English, Japanese, Spanish, etc.) |\n| **Runtime** | 182\u202fminutes (≈3\u202fhours\u202f2\u202fminutes) |\n| **Genre** | Action‑Adventure, Historical Fiction, Epic Drama |\n| **Director** | **S.\u202fS.\u202fRajamouli** – famed for the *Baahubali* franchise |\n| **Screenplay / Story** | S.\u202fS.\u202fRajamouli (story) – co‑written with V.\u202fV.\u202fVinayak and others |\n| **Production Companies** | DV\u202fEntertainments, T-Series Films, Arka Media Works, and others |\n| **Producers** | D.\u202fV.\u202fVishnu\u202fKumar, Ramana\u202fBaba\u202fVenkatesh, Shobu\u202

In [9]:
response = model_with_structure.invoke("Provide details about the movie RRR")
response

Movie(title='RRR', year=2022, director='S. S. Rajamouli', rating=8.5)

In [11]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was realsed")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The rating of the movie")


model_with_structure = model.with_structured_output(Movie,include_raw=True)


response = model_with_structure.invoke("Provide details about the movie RRR")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide details about the movie RRR". They likely want info: director, rating, title, year. Could use the function Movie to get details. The function expects director, rating, title, year. We can call function with those values. We need to fill in correct data for RRR: It\'s an Indian Telugu-language epic action drama directed by S. S. Rajamouli, released 2022, rating? Could be IMDb rating ~8.0. Provide details. Use function.', 'tool_calls': [{'id': 'fc_82527972-e6a6-4155-b10e-fc2e62047e69', 'function': {'arguments': '{"director":"S. S. Rajamouli","rating":8,"title":"RRR","year":2022}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 163, 'prompt_tokens': 158, 'total_tokens': 321, 'completion_time': 0.338017097, 'completion_tokens_details': {'reasoning_tokens': 106}, 'prompt_time': 0.010892371, 'prompt_tokens_details': None, 'queue_time': 0.214032536, 

Nested Structure

In [13]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie RRR")
response

MovieDetails(title='RRR', year=2022, cast=[Actor(name='N. T. Rama Rao Jr.', role='Komaram Bheem'), Actor(name='Ram Charan', role='Alluri Sitarama Raju'), Actor(name='Alia Bhatt', role='Sita'), Actor(name='Ajay Devgn', role='Sukumar'), Actor(name='Olivia Morris', role='Jennifer')], genres=['Action', 'Drama', 'Historical', 'Adventure'], budget=72000000.0)

TypedDict


TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

TypedDict in Python is used to define the expected structure of a dictionary, including the names and data types of its keys.

It is useful when working with dictionaries in Python, LangChain, and LLM responses.

important: TypedDict provides type information for static type checkers and tools. It does not automatically validate dictionary values at runtime.

In [14]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [15]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Clark Gregg', 'role': 'Phil Coulson'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'},
  {'name': 'Stellan Skarsgård', 'role': 'Erik Selvig'},
  {'name': 'Gwyneth Paltrow', 'role': 'Pepper Potts'},
  {'name': 'Paul Bettany', 'role': 'J.A.R.V.I.S. (voice)'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [16]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

DataClass:

A dataclass in Python is a convenient way to create a class that mainly stores data.

It automatically generates things like __init__(), __repr__(), and comparisons for you, so you don't have to write them manually.

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

For LLM structured output, Pydantic is often the best choice when you want strong validation, while TypedDict and dataclasses are useful when you mainly need a defined output schema.

TypedDict:"My data should look like this dictionary."  {"title": "Inception", "year": 2010}

Dataclass:"I want an object that stores this data."  movie.title  movie.year

Pydantic:"I want an object that stores this data AND validates it."  movie.title  movie.year

In [17]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model("groq:openai/gpt-oss-120b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.11', 'langchain': '1.4.6'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002BBF3F4AA20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002BBF3F48E30>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [18]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='d4a8187a-b848-4fb5-827a-9af46c0b5378'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'The user wants to extract contact info from the given text. We need to output JSON according to the ContactInfo schema. Must be compact JSON (no pretty printing). Include fields: name, email, phone. All required. Provide values: "John Doe", "john@example.com", "(555) 123-4567". Ensure it\'s valid JSON. Output only final JSON object.\n\nThus: {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}'}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 238, 'total_tokens': 376, 'completion_time': 0.296543997, 'completion_tokens_details': {'reasoning_tokens': 103}, 'prompt_time': 0.010886141, 'prompt_tokens_details': None

In [19]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [20]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [21]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')